In [ ]:
!pip install -q transformers peft bitsandbytes accelerate trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,TrainingArguments,)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
print(f"GPU available: {torch.cuda.is_available()}")

GPU available: True


In [ ]:
train_ds = "/content/drive/MyDrive/train.json"

In [ ]:
with open(train_ds, "r") as f:
    raw_data = json.load(f)

print(len(raw_data))
print(raw_data[0])

17820
{'instruction': 'Number each line in "foobar" as right-justified zero padded to a width of 9', 'output': 'nl -nrz -w9 foobar'}


In [ ]:
def format_prompt(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Output:\n{example['output']}"
    }


dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_prompt)

print(f"Total examples: {len(dataset)}")

Map:   0%|          | 0/17820 [00:00<?, ? examples/s]

Total examples: 17820


In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
model_name = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True,)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [ ]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules = ["q_proj","k_proj", "v_proj","o_proj",]
)

In [ ]:
output_dir = "/content/drive/MyDrive/phi3.5-mini-finetuned"
repo_id = "llhax/phi3.5-mini-finetuned"

sft_config = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text",
    eval_strategy="no",
    max_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    logging_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    report_to="none",
    gradient_checkpointing=False,
    push_to_hub=True,
    hub_model_id=repo_id,
    hub_strategy="checkpoint",
    save_steps=100,
    hub_token=HF_TOKEN
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)

Adding EOS to train dataset:   0%|          | 0/17820 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/17820 [00:00<?, ? examples/s]

In [ ]:
import os

output_dir = "/content/drive/MyDrive/phi3.5-mini-finetuned"

checkpoints = [
    d for d in os.listdir(output_dir)
    if d.startswith("checkpoint-")
] if os.path.isdir(output_dir) else []

if checkpoints:
    latest = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
    latest_path = os.path.join(output_dir, latest)
    print(f"Resuming from: {latest_path}")
    trainer.train(resume_from_checkpoint=latest_path)
else:
    print("No checkpoint found")
    trainer.train()